# Packed-prefix check, then result-line-only KV mix, plan4_dev, Kaggle T4

Same Tesla T4. Do not switch GPUs. Same Qwen3-4B revision, FP16, greedy decoding.

`plan4_dev` only. Do **not** load `plan4_hold`. Do not resume kvswap, boundary, recon, iface, support, or hold. Do not regenerate A.

The previous mix replaced the whole compact cache, including the first four tokens and the newline.

**Cell 4** (`--phase check`) compares those prefix positions on all 60 unique A sheets. If they are exactly identical, stop: the earlier finding already applies to result-line keys.

If they differ, **cell 5** (`--phase compare`) repeats four conditions with one shared reconstructed prefix, replacing only result-line keys or values:

1. **a_k_a_v** — A's result keys and values.
2. **recon_k_a_v** — reconstructed result keys, A's result values.
3. **a_k_recon_v** — A's result keys, reconstructed result values.
4. **reconstruct** — reconstructed result keys and values (full reconstruction).

Tokens, positions, framing, and decoding stay unchanged. Do not train an adapter yet.

Download `/kaggle/working/results_stage_plan4_kvresult`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_kvresult.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_kvresult.py").is_file() and (cand / "data" / "plan4_dev.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_kvresult.py and data/plan4_dev.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input"):
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    elif not (dest / "src" / "stage_plan4_kvresult.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
    os.chdir(REPO)
else:
    os.chdir(REPO)

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
art = REPO / "results_stage_plan4_iface" / "a_artifacts.jsonl"
print("frozen A artifacts", art.is_file(), art)

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_kvresult")
print("Run check first. Only compare if the packed prefix is not identical.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_kvresult", "--split", "plan4_dev", "--phase", "check"])
print("plan4 kvresult check exit", rc)
path = WORK / "results_stage_plan4_kvresult" / "check_summary.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_kvresult" / "run" / "check_summary.json"
if path.is_file():
    data = json.loads(path.read_text(encoding="utf-8"))
    slim = {k: data.get(k) for k in ("n", "all_passed", "prefix_identical", "prefix", "lead")}
    print(json.dumps(slim, indent=2)[:16000])

In [ ]:
src = WORK / "results_stage_plan4_kvresult" / "check_summary.json"
if not src.is_file():
    src = REPO / "results" / "stage_plan4_kvresult" / "run" / "check_summary.json"
check = json.loads(src.read_text(encoding="utf-8"))
if not check.get("all_passed"):
    raise RuntimeError("Check failed. Do not run compare.")
if check.get("prefix_identical"):
    print("Prefix tensors are exactly identical. Skip compare.")
else:
    rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_kvresult", "--split", "plan4_dev", "--phase", "compare"])
    print("plan4 kvresult compare exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_kvresult"
if not (src / "summary.json").is_file() and not (src / "check_summary.json").is_file():
    src = REPO / "results" / "stage_plan4_kvresult" / "run"
print("reading", src)
for name in ("check_summary.json", "summary.json"):
    path = src / name
    if path.is_file():
        data = json.loads(path.read_text(encoding="utf-8"))
        print("\n====", name, "====")
        keep = {k: data.get(k) for k in ("n", "n_rows", "all_passed", "prefix_identical", "prefix", "receiver", "family_clustered_eligibility", "lead") if k in data or data.get(k) is not None}
        print(json.dumps(keep, indent=2)[:24000])